## Import

In [ ]:
import sys, os
path2add = os.path.normpath(os.path.abspath(os.path.join(os.path.curdir, os.path.pardir)))
print(path2add)

if (not (path2add in sys.path)) :
    sys.path.append(path2add)

In [ ]:
import BeamTestHelpers as helper
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import hist
import yaml
import mplhep as hep
hep.style.use('CMS')

from itertools import combinations

## Specify track information

In [ ]:
with open("../board_configs_yaml/DESY_TB_2026May_CE.yaml", "r") as file:
    fig_configs = yaml.safe_load(file)

print(fig_configs.keys())

given_run = 'run26'
selected_fig_config = fig_configs[given_run]

for id in selected_fig_config.keys():
    selected_fig_config[id]['title'] = f"{selected_fig_config[id]['name']} HV{selected_fig_config[id]['HV']}V OS:{selected_fig_config[id]['offset']}"

    try:
        selected_fig_config[id]['title'] += f' {selected_fig_config[id]['angle']}deg'
    except:
        pass

    try:
        selected_fig_config[id]['title'] += f' temp:{selected_fig_config[id]['temp']}'
    except:
        pass

    try:
        selected_fig_config[id]['title'] += f' IRRAD:{selected_fig_config[id]['irrad']}'
    except:
        pass

    try:
        selected_fig_config[id]['title'] += f' {selected_fig_config[id]['note']}'
    except:
        pass

selected_fig_config

## Set output directory depending on TB campaign

In [ ]:
# current_dir = Path('./')
# output_mother_dir = current_dir / 'etroc_TB_figs'
# output_mother_dir.mkdir(exist_ok=True)

# ### Now you need change the directory name per campaign
# ### Naming rule is this:
# ### <TB location>_TB_MonthYear
# ### E.g. desy_TB_Apr2024, cern_TB_Sep2023, fnal_TB_Jul2024

# output_campaign_dir = output_mother_dir / 'desy_TB_Jun2024'
# # output_campaign_dir.mkdir(exist_ok=True)

## Read track file

In [ ]:
df = pd.read_parquet('./desy_2026May/track_t-R8C8_d-R9C8_r-R8C8.parquet')
df

## Make 1D TDC hists

In [ ]:
h_track = helper.return_hist_pivot(input_df=df, board_info=selected_fig_config, hist_bins=[100, 128, 128])

chip_figtitles = [val['title'] for _, val in selected_fig_config.items()]
helper.plot_1d_TDC_histograms(input_hist=h_track, tb_loc='desy', fig_tag=chip_figtitles, slide_friendly=True)

## Apply TDC cuts

In [ ]:
tot_cuts = {}

# ids = ['dut', 'ref', 'extra']
ids = ['trig', 'dut', 'ref']


for idx in ids:
    lower_bound = df[f'tot_{idx}'].quantile(0.01)
    upper_bound = df[f'tot_{idx}'].quantile(0.96)
    tot_cuts[idx] = [round(lower_bound), round(upper_bound)]

tot_cuts

In [ ]:
def get_robust_range(data, mad_factor=5.0, sigma_factor=3.0, max_iterations=10):
    """
    Determines a robust cut range with an explicit convergence check.
    """
    clean_data = data.dropna().values
    if len(clean_data) == 0:
        return None, None

    # Step 1: Initial MAD-based Pass
    median = np.median(clean_data)
    mad = np.median(np.abs(clean_data - median)) * 1.4826

    initial_mask = (clean_data > (median - mad_factor * mad)) & \
                   (clean_data < (median + mad_factor * mad))
    final_data = clean_data[initial_mask]

    # Step 2: Iterative Sigma Clipping with Convergence
    prev_count = len(final_data)

    for i in range(max_iterations):
        if len(final_data) < 2:
            break

        mean = np.mean(final_data)
        std = np.std(final_data)

        lower = mean - sigma_factor * std
        upper = mean + sigma_factor * std

        # Apply the new cut
        mask = (final_data >= lower) & (final_data <= upper)
        final_data = final_data[mask]

        # --- CONVERGENCE CHECK ---
        current_count = len(final_data)

        # Option A: Check if the number of points stopped changing
        if current_count == prev_count:
            print(f"Converged after {i+1} iterations.")
            break

        prev_count = current_count

    if len(final_data) == 0:
        return None, None

    return np.min(final_data), np.max(final_data)

# Usage Example:
# df = pd.read_csv("your_data.csv")
results = {}

for idx in ids:
    low, high = get_robust_range(df[f'tot_{idx}'], sigma_factor=3.0)
    results[f'{idx}'] = (low, high)
    print(f"{f'tot_{idx}'}: Cut range is [{low:.2f}, {high:.2f}]")

In [ ]:
for idx in ids:

    plt.figure(figsize=(9,9))
    plt.hist(df[f'tot_{idx}'], bins=256, range=(0,512), histtype='step', lw=2)
    plt.axvline(x=results[idx][0], color='black', linestyle='dashed')
    plt.axvline(x=results[idx][1], color='black', linestyle='dashed')
    plt.xlabel(idx)
    plt.xlim(None, 200)

In [ ]:
## Selecting good hits with TDC cuts
tdc_cuts = {}
for idx in ids:
    if idx == 0:
        tdc_cuts[idx] = [0, 1100, 0, 800, results[idx][0], results[idx][1]]
    else:
        tdc_cuts[idx] = [0, 1100, 0, 800, results[idx][0], results[idx][1]]

track_tmp_df = helper.tdc_event_selection_pivot(df, tdc_cuts_dict=tdc_cuts)

In [ ]:
track_tmp_df

## Make TOA correlation plots

In [ ]:
helper.plot_TOA_correlation(track_tmp_df, board_role1='trig', board_role2='dut', board_names=['Trig', 'DUT'], boundary_cut=3, draw_boundary=True, tb_loc='desy')#, save_mother_dir=output_campaign_dir)
helper.plot_TOA_correlation(track_tmp_df, board_role1='trig', board_role2='ref', board_names=['Trig', 'Ref'], boundary_cut=3, draw_boundary=True, tb_loc='desy')#, save_mother_dir=output_campaign_dir)
helper.plot_TOA_correlation(track_tmp_df, board_role1='dut', board_role2='ref', board_names=['DUT', 'Ref'], boundary_cut=3, draw_boundary=True, tb_loc='desy')#, save_mother_dir=output_campaign_dir)

In [ ]:
_, dist1 = helper.return_TOA_correlation_param(track_tmp_df, 'trig', 'dut')
_, dist2 = helper.return_TOA_correlation_param(track_tmp_df, 'trig', 'ref')
_, dist3 = helper.return_TOA_correlation_param(track_tmp_df, 'dut', 'ref')

condition1 = np.abs(dist1) < 3*np.std(dist1)
condition2 = np.abs(dist2) < 3*np.std(dist2)
condition3 = np.abs(dist3) < 3*np.std(dist3)

distance_condition = condition1 & condition2 & condition3

In [ ]:
analysis_df = track_tmp_df[distance_condition].reset_index(drop=True)
analysis_df = analysis_df.reset_index().rename(columns={'index':'evt'})
analysis_df

In [ ]:
h_track = helper.return_hist_pivot(input_df=analysis_df, board_info=selected_fig_config, hist_bins=[100, 128, 128])

chip_figtitles = [val['title'] for _, val in selected_fig_config.items()]
helper.plot_1d_TDC_histograms(input_hist=h_track, tb_loc='irrad', fig_tag=chip_figtitles, slide_friendly=True)

## Random sampling iteration

In [ ]:
random_df = analysis_df.sample(frac=0.75, ignore_index=True)
random_df

## Make dataframe in time

In [ ]:
# board_to_analyze = ['dut', 'ref', 'extra']
board_to_analyze = ['trig', 'dut', 'ref']


d = {}

for idx in board_to_analyze:
    bins = 3.125/random_df[f'cal_{idx}'].mean()
    d[f'toa_{str(idx)}'] = (12.5 - random_df[f'toa_{idx}'] * bins)*1e3
    d[f'tot_{str(idx)}'] = ((2*random_df[f'tot_{idx}'] - np.floor(random_df[f'tot_{idx}']/32)) * bins)*1e3

df_in_time = pd.DataFrame(data=d)

In [ ]:
df_in_time

## TOA hists before TWC

In [ ]:
h_before_twc_toa1 = hist.Hist(hist.axis.Regular(50, 0, 14000, name=f'toa_{board_to_analyze[0]}', label=f'toa_{board_to_analyze[0]}'))
h_before_twc_toa2 = hist.Hist(hist.axis.Regular(50, 0, 14000, name=f'toa_{board_to_analyze[1]}', label=f'toa_{board_to_analyze[1]}'))
h_before_twc_toa3 = hist.Hist(hist.axis.Regular(50, 0, 14000, name=f'toa_{board_to_analyze[2]}', label=f'toa_{board_to_analyze[2]}'))

h_before_twc_toa1.fill(df_in_time[f'toa_{board_to_analyze[0]}'])
h_before_twc_toa2.fill(df_in_time[f'toa_{board_to_analyze[1]}'])
h_before_twc_toa3.fill(df_in_time[f'toa_{board_to_analyze[2]}'])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(25, 10))
yerr_bool = np.array(h_before_twc_toa1.values(), dtype=bool)
hep.histplot(h_before_twc_toa1, ax=axes[0], yerr=yerr_bool)
axes[0].set_xlabel('TOA1')

yerr_bool = np.array(h_before_twc_toa2.values(), dtype=bool)
hep.histplot(h_before_twc_toa2, ax=axes[1], yerr=yerr_bool)
axes[1].set_xlabel('TOA2')

yerr_bool = np.array(h_before_twc_toa3.values(), dtype=bool)
hep.histplot(h_before_twc_toa3, ax=axes[2], yerr=yerr_bool)
axes[2].set_xlabel('TOA3')

fig.tight_layout()

### Delta TOA hists before TWC

In [ ]:
board_list = [1, 2, 3]

del_toa_b0 = (0.5*(df_in_time[f'toa_{board_to_analyze[1]}'] + df_in_time[f'toa_{board_to_analyze[2]}']) - df_in_time[f'toa_{board_to_analyze[0]}'])
del_toa_b1 = (0.5*(df_in_time[f'toa_{board_to_analyze[0]}'] + df_in_time[f'toa_{board_to_analyze[2]}']) - df_in_time[f'toa_{board_to_analyze[1]}'])
del_toa_b2 = (0.5*(df_in_time[f'toa_{board_to_analyze[0]}'] + df_in_time[f'toa_{board_to_analyze[1]}']) - df_in_time[f'toa_{board_to_analyze[2]}'])

In [ ]:
h_before_twc_delta_toa1 = hist.Hist(hist.axis.Regular(50, -2500, 2500, name=f'toa_{board_to_analyze[0]}', label=f'toa_{board_to_analyze[0]}'))
h_before_twc_delta_toa2 = hist.Hist(hist.axis.Regular(50, -2500, 2500, name=f'toa_{board_to_analyze[1]}', label=f'toa_{board_to_analyze[1]}'))
h_before_twc_delta_toa3 = hist.Hist(hist.axis.Regular(50, -2500, 2500, name=f'toa_{board_to_analyze[2]}', label=f'toa_{board_to_analyze[2]}'))

h_before_twc_delta_toa1.fill(del_toa_b0)
h_before_twc_delta_toa2.fill(del_toa_b1)
h_before_twc_delta_toa3.fill(del_toa_b2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(25, 10))
yerr_bool = np.array(h_before_twc_delta_toa1.values(), dtype=bool)
hep.histplot(h_before_twc_delta_toa1, ax=axes[0], yerr=yerr_bool)
axes[0].set_xlabel('0.5*(TOA2+TOA3)-TOA1', fontsize=15)


yerr_bool = np.array(h_before_twc_delta_toa2.values(), dtype=bool)
hep.histplot(h_before_twc_delta_toa2, ax=axes[1], yerr=yerr_bool)
axes[1].set_xlabel('0.5*(TOA1+TOA3)-TOA2', fontsize=15)


yerr_bool = np.array(h_before_twc_delta_toa3.values(), dtype=bool)
hep.histplot(h_before_twc_delta_toa3, ax=axes[2], yerr=yerr_bool)
axes[2].set_xlabel('0.5*(TOA2+TOA3)-TOA1', fontsize=15)

fig.tight_layout()

### Pairwise delta TOA before TWC

In [ ]:
pre_diffs = {f"{r1}-{r2}": (df_in_time[f'toa_{r1}'] - df_in_time[f'toa_{r2}']).to_numpy() for r1, r2 in combinations(board_to_analyze, 2)}
pre_diffs

In [ ]:
fit_sigmas = {}
for pair, data in pre_diffs.items():
    fwhm, ks = helper.fit_gmm_and_get_fwhm(data, pair, plot_result=True, plot_cdf=True)
    fit_sigmas[pair] = fwhm / 2.355

In [ ]:
res = helper.calculate_resolution_from_fit(fit_sigmas, board_to_analyze)
res

### Investigate TWC

In [ ]:
helper.plot_TWC(input_df=df_in_time, tb_loc='desy', board_ids=board_to_analyze, poly_order=2)#, save_mother_dir=output_campaign_dir)

In [ ]:
# helper.plot_TWC_with_residual_check(df_in_time, board_to_analyze, tb_loc='desy')

### Iterative TWC step by step

In [ ]:
corr_toas = helper.three_board_iterative_timewalk_correction(df_in_time, board_to_analyze)
diffs = {f"{r1}-{r2}": corr_toas[r1] - corr_toas[r2] for r1, r2 in combinations(board_to_analyze, 2)}

In [ ]:
corr_toas

In [ ]:
helper.plot_TWC(input_df=df_in_time, tb_loc='desy', board_ids=board_to_analyze, poly_order=2, corr_toas=corr_toas)#, save_mother_dir=output_campaign_dir)

In [ ]:
pd.DataFrame(corr_toas)

In [ ]:
# 1. Extract only the TOT columns from your original dataframe
tot_df = df_in_time.filter(like='tot_')

# 2. Convert the corrected TOAs to a DataFrame and add the 'toa_' prefix
# to ensure it matches your analysis format (e.g., 'toa_dut')
corr_df = pd.DataFrame(corr_toas).add_prefix('toa_')

# 3. Concatenate them side-by-side
df_corrected_final = pd.concat([tot_df, corr_df], axis=1)
df_corrected_final

In [ ]:
helper.plot_TWC_with_residual_check(df_corrected_final, board_to_analyze, tb_loc='desy')

In [ ]:
fit_sigmas = {}
for pair, data in diffs.items():
    fwhm, ks = helper.fit_gmm_and_get_fwhm(data, pair, plot_result=True, plot_cdf=True)
    fit_sigmas[pair] = fwhm / 2.355

In [ ]:
res = helper.calculate_resolution_from_fit(fit_sigmas, board_to_analyze)
res